In [6]:
from torch import nn
import torch
from torchvision import transforms,datasets
from torch.utils.data import DataLoader

In [7]:
device=torch.device( "cuda" if torch.cuda.is_available() else "cpu")

transform=transforms.Compose(transforms.ToTensor())
train_dataset=datasets.MNIST(transform=transform,download=True,train=True,root="data")
test_dataset=datasets.MNIST(transform=transform,download=True,train=False,root="data")
train_loader=DataLoader(train_dataset,shuffle=True,batch_size=32)
test_loader=DataLoader(test_dataset,shuffle=False,batch_size=32)

In [12]:
class GRU(nn.Module):
    def __init__(self,hidden_size,input_size):
        super().__init__()
        self.hidden_size=hidden_size
        self.input_size=input_size
        self.candidate_gate=nn.Linear(self.hidden_size+self.input_size,self.hidden_size)
        self.reset_gate=nn.Linear(self.hidden_size+self.input_size,self.hidden_size)
        self.update_gate=nn.Linear(self.hidden_size+self.input_size,self.hidden_size)
    def forward(self,previous_layer,x):
        combined=torch.cat((x,previous_layer),dim=1)
        update=torch.sigmoid(self.update_gate(combined))
        reset=torch.sigmoid(self.reset_gate(combined))
        candidate=torch.cat((x,previous_layer*reset),dim=1)
        candidate=torch.tanh(self.candidate_gate(candidate))
        new_hidden=(update*candidate)+(1-update)*previous_layer
        return new_hidden


In [15]:
model=GRU(5,5)
sequence=torch.rand(1,5,5)
hidden=torch.zeros(1,5)
for i in range(5):
    x=sequence[:,i,:]
    hidden=model(hidden,x)
    print(hidden,"\n")

tensor([[ 0.2569, -0.1463,  0.2959, -0.3147,  0.1729]], grad_fn=<AddBackward0>) 

tensor([[ 0.2873, -0.2333,  0.4088, -0.4272,  0.1803]], grad_fn=<AddBackward0>) 

tensor([[ 0.3900, -0.3523,  0.4846, -0.5331,  0.2347]], grad_fn=<AddBackward0>) 

tensor([[ 0.4470, -0.3723,  0.5182, -0.6084,  0.2175]], grad_fn=<AddBackward0>) 

tensor([[ 0.5134, -0.4373,  0.5585, -0.6804,  0.2717]], grad_fn=<AddBackward0>) 

